# Fine-tune Qwen2.5-Coder-1.5B (LoRA) para text-to-SQL

Notebook Kaggle (GPU T4, Internet ligada) usado no `multistore-ai-assistant`.

**Inputs (datasets Kaggle):**
- `multistore-ft`: `finetune_train.jsonl`, `finetune_valid.jsonl` (gerados a partir de `data/finetune_pairs.jsonl`)
- `multistore-eval`: `eval_prompts.jsonl` (gerado com `python -m multistore_ai_assistant.finetune.export_eval`)

**Outputs (`/kaggle/working`):** `qwen-sql-lora.zip`, `preds_qwen_ft.jsonl`, `preds_qwen_base.jsonl`.
As previsões são pontuadas localmente com `python -m multistore_ai_assistant.finetune.score`.

Resultado: Qwen 1.5B afinado 18/40 = 45 %, Qwen 1.5B base 10/40 = 25 % (Groq gpt-oss-120b: 40/40).

## 1. Instalação (uma só GPU, o Unsloth não usa as duas)

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
!pip install -q unsloth

## 2. Ficheiros de input

In [ ]:
import glob, json
TRAIN = glob.glob("/kaggle/input/**/finetune_train.jsonl", recursive=True)[0]
VALID = glob.glob("/kaggle/input/**/finetune_valid.jsonl", recursive=True)[0]
EVAL  = glob.glob("/kaggle/input/**/eval_prompts.jsonl", recursive=True)[0]
print(TRAIN, VALID, EVAL)

## 3. Modelo em 4 bits + LoRA

In [ ]:
from unsloth import FastLanguageModel

BASE = "unsloth/Qwen2.5-Coder-1.5B-Instruct"
MAX_LEN = 3072
model, tokenizer = FastLanguageModel.from_pretrained(
    BASE, max_seq_length=MAX_LEN, load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42,
)

## 4. Dados (chat template do Qwen)

In [ ]:
from datasets import load_dataset

data = load_dataset("json", data_files={"train": TRAIN, "valid": VALID})

def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)}

data = data.map(to_text)
print(data)
print(data["train"][0]["text"][-300:])

## 5. Treino (só nas respostas do assistente)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=data["train"], eval_dataset=data["valid"],
    args=SFTConfig(
        dataset_text_field="text", max_seq_length=MAX_LEN, packing=True,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        num_train_epochs=2, learning_rate=2e-4, optim="adamw_8bit",
        logging_steps=5, output_dir="/kaggle/working/out", report_to="none", seed=42,
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
trainer.train()

## 6. Teste rápido numa pergunta de validação

In [ ]:
FastLanguageModel.for_inference(model)
msgs = data["valid"][0]["messages"]
prompt = tokenizer.apply_chat_template(msgs[:-1], tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
out = model.generate(**inputs, max_new_tokens=300, do_sample=False)
print("PERGUNTA:", msgs[1]["content"])
print("ESPERADO:", msgs[2]["content"])
print("GERADO:  ", tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))

## 7. Guardar o adapter LoRA

In [ ]:
model.save_pretrained("/kaggle/working/qwen-sql-lora")
tokenizer.save_pretrained("/kaggle/working/qwen-sql-lora")
!cd /kaggle/working && zip -qr qwen-sql-lora.zip qwen-sql-lora && ls -lh

## 8. Gerar SQL para as 40 perguntas do eval (modelo afinado)

In [ ]:
prompts = [json.loads(l) for l in open(EVAL, encoding="utf-8")]

def gerar(model, tokenizer, out_path):
    FastLanguageModel.for_inference(model)
    with open(out_path, "w", encoding="utf-8") as f:
        for p in prompts:
            text = tokenizer.apply_chat_template(p["messages"], tokenize=False, add_generation_prompt=True)
            inputs = tokenizer(text, return_tensors="pt").to("cuda")
            out = model.generate(**inputs, max_new_tokens=400, do_sample=False)
            sql = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()
            f.write(json.dumps({"n": p["n"], "sql": sql}, ensure_ascii=False) + "\n")
            print(p["n"], sql[:80].replace("\n", " "))

gerar(model, tokenizer, "/kaggle/working/preds_qwen_ft.jsonl")

## 9. Gerar SQL com o modelo base (sem fine-tune), para comparar

In [ ]:
base_model, base_tok = FastLanguageModel.from_pretrained(
    BASE, max_seq_length=MAX_LEN, load_in_4bit=True,
)
gerar(base_model, base_tok, "/kaggle/working/preds_qwen_base.jsonl")